In [ ]:
import ClimaComms
ClimaComms.@import_required_backends
import ClimaUtilities
import ClimaUtilities.TimeManager: ITime, date
import ClimaUtilities.TimeVaryingInputs:TimeVaryingInput, LinearInterpolation, PeriodicCalendar
import ClimaUtilities.ClimaArtifacts: @clima_artifact
import ClimaParams as CP

using ClimaCore
using ClimaLand
using ClimaLand.Snow
using ClimaLand.Soil
using ClimaLand.Canopy

import ClimaLand.Parameters as LP
import ClimaLand.LandSimVis as LandSimVis
import ClimaLand.Simulations: LandSimulation, solve!
import ClimaDiagnostics

using Dates
using NCDatasets

using CairoMakie, GeoMakie, ClimaAnalysis

const FT = Float64;
context = ClimaComms.context()
ClimaComms.init(context)
device = ClimaComms.device()
device_suffix = device isa ClimaComms.CPUSingleThreaded ? "cpu" : "gpu"
root_path = "single_site"
diagnostics_outdir = joinpath(root_path, "global_diagnostics")
outdir = ClimaUtilities.OutputPathGenerator.generate_output_path(diagnostics_outdir)

In [ ]:
function setup_model(
    ::Type{FT},
    start_date,
    stop_date,
    Δt,
    domain,
    toml_dict,
) where {FT}
    surface_domain = ClimaLand.Domains.obtain_surface_domain(domain)
    surface_space = domain.space.surface
    # Forcing data - high resolution
    atmos, radiation = ClimaLand.prescribed_forcing_era5(
        start_date,
        stop_date,
        surface_space,
        toml_dict,
        FT;
        max_wind_speed = 25.0,
        context,
        use_lowres_forcing = true,
    )
    forcing = (; atmos, radiation)

    # Read in LAI from MODIS data
    LAI = ClimaLand.Canopy.prescribed_lai_modis(
        surface_space,
        start_date,
        stop_date,
    )

    land = LandModel{FT}(forcing, toml_dict, domain, Δt;)

    # default_soil = land.soil.parameters

    # custom_soil_params = Soil.EnergyHydrologyParameters(
    #     toml_dict;
    #     hydrology_cm = default_soil.hydrology_cm, # 保留默认的 van Genuchten 水力特征[cite: 9]
    #     ν = default_soil.ν,                       # 保留默认的孔隙度[cite: 9]
    #     K_sat = default_soil.K_sat,               # 保留默认的饱和导水率[cite: 9]
    #     S_s = default_soil.S_s,                   # 保留默认的比储水率[cite: 9]
    #     θ_r = default_soil.θ_r,                   # 保留默认的残余含水率[cite: 9]
    #     ν_ss_om = default_soil.ν_ss_om,           # 保留有机质比例[cite: 9]
    #     ν_ss_quartz = default_soil.ν_ss_quartz,   # 保留石英比例[cite: 9]
    #     ν_ss_gravel = default_soil.ν_ss_gravel,   # 保留砾石比例[cite: 9]
    #     emissivity = FT(0.5)                      # <--- 仅覆盖你想要修改的地表发射率[cite: 3, 10]
    #     )

    # land = LandModel{FT}(forcing, toml_dict, domain, Δt; soil_params = custom_soil_params)
    
    return land
end

In [ ]:
start_date = DateTime("2008-03-01")
stop_date = DateTime("2010-03-01")
Δt = 450.0
# longlat = FT.((-156.0, 71.0))
longlat = FT.((100.9, 38.0))
zlim = FT.((-15, 0))
nelements = 150
dz_tuple = FT.((3, 0.05))
domain = ClimaLand.Domains.Column(; zlim, longlat, nelements, dz_tuple);
toml_dict = LP.create_toml_dict(FT)
# out_writer = ClimaDiagnostics.Writers.NetCDFWriter(domain.space.subsurface,outdir;start_date,)
out_writer = ClimaDiagnostics.Writers.DictWriter()

model = setup_model(FT, start_date, stop_date, Δt, domain, toml_dict);

saveat = Hour(24)
saving_cb = ClimaLand.NonInterpSavingCallback(start_date, stop_date, saveat) # 创建保存回调
sv = saving_cb.affect!.saved_values # 获取保存数据的句柄 (引用)

diagnostics = ClimaLand.default_diagnostics(
    model,
    start_date;
    output_writer = out_writer,
    reduction_period = :daily,
    reduction_type = :average,
    output_vars = [
        "shf",
        "lhf",
        "trans",
        "swu",
        "lwu",
        "sr",
        "ssr",
        "precip",
        "et",
        "lai",
        "tsoil"
    ],
);
simulation =
    LandSimulation(start_date, stop_date, Δt, model; 
    outdir, 
    updateat = Δt,
    solver_kwargs = (; saveat),
    user_callbacks = (saving_cb,), 
    diagnostics);

In [ ]:
@info "Run: Global Soil-Canopy-Snow-SoilCO2 Model"
@info "Resolution: $(domain.nelements)"
@info "Timestep: $Δt s"
@info "Start Date: $start_date"
@info "Stop Date: $stop_date"
CP.log_parameter_information(toml_dict, joinpath(root_path, "parameters.toml"))
sol = ClimaLand.Simulations.solve!(simulation);

In [ ]:
LandSimVis.make_timeseries(
    simulation;
    savedir = root_path,
    short_names = ["lai","tsoil"],
)

In [ ]:
soil_depth = parent(ClimaCore.Fields.coordinate_field(domain.space.subsurface).z)[:];

t_dates = date.(sol.t)

# 1. 将时间转换为纯数值（天数），避开 DateTime 导致的崩溃
t_days = Float64.(sol.t) ./ (24 * 3600)

T = [parent(sv.saveval[k].soil.T)[:] for k in 1:length(sol.t)];

T_matrix = permutedims(reduce(hcat, T)) .- 273.15

fig = Figure(size = (800, 400), fontsize = 16)
ax = Axis(fig[1, 1], xlabel = "Days since start", ylabel = "Soil Depth (m)")
hm = CairoMakie.heatmap!(
    ax, 
    t_days, 
    soil_depth, 
    T_matrix, 
    colorrange = (-20,20),
    colormap = Reverse(:RdYlBu)
)
CairoMakie.contour!(ax, t_days, soil_depth, T_matrix, levels = [0], color = :black, linewidth = 2)
Colorbar(fig[1, 2], hm, label = "Temperature (°C)")
CairoMakie.ylims!(ax, -5, 0)
fig